# Part 2a - Co-teaching (filtering during training)
No cleaning before training. Two networks (one per T4) are trained simultaneously; in every
mini-batch each network ranks the samples by its own loss, keeps the `(1 - forget)` fraction with the
smallest loss and **gives them to the peer network** for its update (errors of one network are not
fed back into itself). `forget` is 0 during a short warm-up, then ramps linearly over 10 epochs to the
noise rate **estimated from the data** (2-component GMM on the losses at the end of warm-up).
The submission averages the two (EMA) networks.


In [ ]:
%%writefile nl.py
#!/usr/bin/env python

import argparse
import copy
import glob
import json
import math
import os
import pickle
import random
import time
from contextlib import contextmanager
from typing import Callable, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset

NUM_CLASSES = 100
MEAN = (0.5071, 0.4865, 0.4409)
STD = (0.2673, 0.2564, 0.2762)
DEFAULT_ROOT = ('/kaggle/input/competitions/dl-gen-ai-2026-fii-competition-3/'
                'fii-atnn-2024-project-noisy-cifar-100')
NOISE_FILE = 'CIFAR-100-noisy.npz'
ACTIONS = ('keep', 'relabel', 'drop')


def log(msg):
    print(time.strftime('%H:%M:%S'), msg, flush=True)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed % (2 ** 32))
    torch.manual_seed(seed)


def _jsonable(o):
    if isinstance(o, (np.generic,)):
        return o.item()
    if isinstance(o, np.ndarray):
        return o.tolist()
    if torch.is_tensor(o):
        return o.tolist()
    return str(o)


class CIFAR100_noisy_fine(Dataset):

    def __init__(
        self, root: str, train: bool, transform: Optional[Callable], download: bool
    ):
        from torchvision.datasets import CIFAR100
        cifar100 = CIFAR100(
            root=root, train=train, transform=transform, download=download
        )
        data, targets = tuple(zip(*cifar100))

        if train:
            noisy_label_file = os.path.join(root, "CIFAR-100-noisy.npz")
            if not os.path.isfile(noisy_label_file):
                raise FileNotFoundError(
                    f"{type(self).__name__} need {noisy_label_file} to be used!"
                )

            noise_file = np.load(noisy_label_file)
            if not np.array_equal(noise_file["clean_label"], targets):
                raise RuntimeError("Clean labels do not match!")
            targets = noise_file["noisy_label"]

        self.data = data
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i: int):
        return self.data[i], self.targets[i]


def find_root(root):
    if root and os.path.isfile(os.path.join(root, NOISE_FILE)):
        return root
    hits = sorted(glob.glob(os.path.join('/kaggle/input', '**', NOISE_FILE), recursive=True))
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError(f'{NOISE_FILE} not found under {root!r} or /kaggle/input; '
                            'attach the competition data to the notebook.')


def _class_names(root):
    try:
        with open(os.path.join(root, 'cifar-100-python', 'meta'), 'rb') as f:
            return [str(c) for c in pickle.load(f, encoding='latin1')['fine_label_names']]
    except Exception:
        return [str(i) for i in range(NUM_CLASSES)]


def _synthetic(n_train, seed=0):
    """Tiny learnable fake dataset, only used by the smoke test (NL_SMOKE=<n_train>)."""
    rng = np.random.RandomState(seed)
    protos = rng.randint(0, 256, size=(NUM_CLASSES, 4, 4, 3)).astype(np.float32)

    def make(n):
        y = rng.randint(0, NUM_CLASSES, n)
        x = protos[y].repeat(8, axis=1).repeat(8, axis=2) + rng.normal(0, 50, (n, 32, 32, 3))
        return np.clip(x, 0, 255).astype(np.uint8), y

    x_train, y = make(n_train)
    x_test, _ = make(max(200, n_train // 5))
    y_noisy = y.copy()
    flip = rng.rand(n_train) < 0.4
    y_noisy[flip] = rng.randint(0, NUM_CLASSES, int(flip.sum()))
    return dict(x_train=x_train, y_noisy=y_noisy.astype(np.int64), x_test=x_test,
                classes=[f'c{i}' for i in range(NUM_CLASSES)])


def load_data(root=DEFAULT_ROOT, cache_dir='/tmp/nl_cache'):
    """Returns train images (N,32,32,3 uint8), the NOISY train labels and the test images."""
    smoke = int(os.environ.get('NL_SMOKE', '0'))
    if smoke:
        return _synthetic(smoke)
    cache = os.path.join(cache_dir, 'cifar100n.npz')
    if os.path.isfile(cache):
        z = np.load(cache)
        return dict(x_train=z['x_train'], y_noisy=z['y_noisy'], x_test=z['x_test'],
                    classes=[str(c) for c in z['classes']])
    t0 = time.time()
    root = find_root(root)
    tr = CIFAR100_noisy_fine(root, train=True, transform=None, download=False)
    te = CIFAR100_noisy_fine(root, train=False, transform=None, download=False)
    out = dict(x_train=np.stack([np.asarray(im, dtype=np.uint8) for im in tr.data]),
               y_noisy=np.asarray(tr.targets, dtype=np.int64),
               x_test=np.stack([np.asarray(im, dtype=np.uint8) for im in te.data]),
               classes=_class_names(root))
    del tr, te  # te.targets (the real test labels) is never used
    os.makedirs(cache_dir, exist_ok=True)
    tmp = f'{cache}.{os.getpid()}.tmp.npz'
    np.savez(tmp, x_train=out['x_train'], y_noisy=out['y_noisy'], x_test=out['x_test'],
             classes=np.array(out['classes']))
    os.replace(tmp, cache)
    log(f'data loaded from {root} in {time.time() - t0:.0f}s: train {out["x_train"].shape}, '
        f'test {out["x_test"].shape}')
    return out


def stratified_folds(y, k, seed):
    rng = np.random.RandomState(seed)
    fold = np.empty(len(y), np.int64)
    for c in np.unique(y):
        idx = np.flatnonzero(y == c)
        rng.shuffle(idx)
        fold[idx] = (np.arange(len(idx)) + rng.randint(k)) % k
    return fold


def stratified_subset(idx, y_idx, n, seed):
    """n indices out of idx, stratified by y_idx (used for the noisy holdout)."""
    if n <= 0:
        return np.empty(0, np.int64)
    rng = np.random.RandomState(seed)
    out = []
    for c in np.unique(y_idx):
        cand = idx[y_idx == c]
        take = int(round(n * len(cand) / len(idx)))
        out.append(rng.permutation(cand)[:take])
    return np.sort(np.concatenate(out))


def random_crop_flip(x, pad=4):
    """Per-sample random crop (reflect padding) + horizontal flip, vectorised on the GPU."""
    b, c, h, w = x.shape
    dev = x.device
    xp = F.pad(x, (pad, pad, pad, pad), mode='reflect')
    i = torch.randint(0, 2 * pad + 1, (b,), device=dev)
    j = torch.randint(0, 2 * pad + 1, (b,), device=dev)
    rows = i[:, None] + torch.arange(h, device=dev)[None]
    cols = j[:, None] + torch.arange(w, device=dev)[None]
    flip = torch.rand(b, device=dev) < 0.5
    cols = torch.where(flip[:, None], cols.flip(1), cols)
    return xp[torch.arange(b, device=dev)[:, None, None, None],
              torch.arange(c, device=dev)[None, :, None, None],
              rows[:, None, :, None], cols[:, None, None, :]]


class DeviceData:
    """All images of one split kept as uint8 on the GPU; batches are normalised/augmented there."""

    def __init__(self, x_nhwc, device):
        self.device = device
        self.x = torch.from_numpy(np.ascontiguousarray(x_nhwc)).to(device).permute(0, 3, 1, 2).contiguous()
        self.mean = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
        self.std = torch.tensor(STD, device=device).view(1, 3, 1, 1)

    def __len__(self):
        return self.x.shape[0]

    def batch(self, idx, augment):
        xb = (self.x[idx].float().div_(255) - self.mean) / self.std
        if augment:
            xb = random_crop_flip(xb)
        return xb.contiguous(memory_format=torch.channels_last)


def make_batch(D, idx, T, alpha):
    """T: (N,) long labels or (N,C) soft targets on D.device. Returns mixup-ed inputs/targets."""
    x = D.batch(idx, augment=True)
    t = T[idx]
    if t.dtype == torch.long:
        t = F.one_hot(t, NUM_CLASSES).float()
    if alpha > 0:
        lam = float(np.random.beta(alpha, alpha))
        perm = torch.randperm(x.shape[0], device=x.device)
        x = lam * x + (1 - lam) * x[perm]
        t = lam * t + (1 - lam) * t[perm]
    return x.contiguous(memory_format=torch.channels_last), t


def epoch_batches(pool, bs, steps):
    """`steps` index batches drawn without replacement (cycling if the pool is small)."""
    n = len(pool)
    reps = max(1, math.ceil(steps * bs / n))
    perm = torch.cat([pool[torch.randperm(n, device=pool.device)] for _ in range(reps)])
    for s in range(steps):
        yield perm[s * bs:(s + 1) * bs]


class PreActBlock(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(cin)
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False)
        self.shortcut = (nn.Conv2d(cin, cout, 1, stride, bias=False)
                         if stride != 1 or cin != cout else None)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        sc = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + sc


class PreActResNet18(nn.Module):
    """CIFAR PreAct ResNet-18 (3x3 stem, no max-pool), the standard backbone of the
    noisy-label literature (DivideMix, ELR, ...). Randomly initialised."""

    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        cfg = [(64, 1), (64, 1), (128, 2), (128, 1), (256, 2), (256, 1), (512, 2), (512, 1)]
        blocks, cin = [], 64
        for cout, stride in cfg:
            blocks.append(PreActBlock(cin, cout, stride))
            cin = cout
        self.layers = nn.Sequential(*blocks)
        self.bn = nn.BatchNorm2d(512)
        self.fc = nn.Linear(512, num_classes)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x, feats=False):
        f = F.adaptive_avg_pool2d(F.relu(self.bn(self.layers(self.conv1(x)))), 1).flatten(1)
        logits = self.fc(f)
        return (logits, f) if feats else logits


def newton_schulz(g, steps=5, eps=1e-7):
    """Quintic Newton-Schulz iteration approximating the orthogonal factor U V^T of g (Muon)."""
    a, b, c = 3.4445, -4.7750, 2.0315
    x = g.float()
    transposed = x.size(0) > x.size(1)
    if transposed:
        x = x.T
    x = x / (x.norm() + eps)
    for _ in range(steps):
        A = x @ x.T
        B = b * A + c * (A @ A)
        x = a * x + B @ x
    return x.T if transposed else x


class MuonSGD(torch.optim.Optimizer):
    """Muon (momentum + Newton-Schulz orthogonalisation, decoupled weight decay)"""

    def __init__(self, groups):
        super().__init__(groups, dict(lr=0.02, momentum=0.95, weight_decay=0.0, use_muon=False))

    @torch.no_grad()
    def step(self, closure=None):
        for g in self.param_groups:
            lr, mom, wd = g['lr'], g['momentum'], g['weight_decay']
            for p in g['params']:
                if p.grad is None:
                    continue
                st = self.state[p]
                if 'buf' not in st:
                    st['buf'] = torch.zeros_like(p)
                buf = st['buf']
                if g['use_muon']:
                    buf.mul_(mom).add_(p.grad)
                    u = p.grad.add(buf, alpha=mom)  # Nesterov
                    rows = u.shape[0]
                    cols = u.numel() // rows
                    u = newton_schulz(u.reshape(rows, cols)).view_as(p).to(p.dtype)
                    p.mul_(1 - lr * wd)
                    p.add_(u, alpha=-lr * max(1.0, rows / cols) ** 0.5)
                else:
                    d = p.grad.add(p, alpha=wd) if wd else p.grad
                    buf.mul_(mom).add_(d)
                    p.add_(d.add(buf, alpha=mom), alpha=-lr)


def build_optimizer(model, args):
    if args.opt == 'sgd':
        opt = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, nesterov=True,
                              weight_decay=args.wd)
    elif args.opt == 'muon':
        muon, other = [], []
        for n, p in model.named_parameters():
            (muon if p.ndim == 4 and n != 'conv1.weight' else other).append(p)
        opt = MuonSGD([dict(params=muon, use_muon=True, lr=args.muon_lr, momentum=0.95,
                            weight_decay=args.muon_wd),
                       dict(params=other, use_muon=False, lr=args.lr, momentum=0.9,
                            weight_decay=args.wd)])
    else:
        raise ValueError(args.opt)
    for g in opt.param_groups:
        g['base_lr'] = g['lr']
    return opt


def autocast(device, enabled):
    return torch.autocast(device_type=device.type,
                          dtype=torch.float16 if device.type == 'cuda' else torch.bfloat16,
                          enabled=enabled)


@contextmanager
def frozen_bn_stats(model):
    """BatchNorm still normalises with batch statistics but does not update running stats
    (used for the second, perturbed forward pass of SAM)."""
    saved = []
    for m in model.modules():
        if isinstance(m, nn.modules.batchnorm._BatchNorm):
            saved.append((m, m.momentum))
            m.momentum = 0.0
    try:
        yield
    finally:
        for m, mo in saved:
            m.momentum = mo


class EMA:
    def __init__(self, model, decay):
        self.decay, self.n = decay, 0
        self.model = copy.deepcopy(model).eval()
        for p in self.model.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model):
        self.n += 1
        d = min(self.decay, (1 + self.n) / (10 + self.n))
        torch._foreach_lerp_(list(self.model.parameters()), list(model.parameters()), 1 - d)
        for eb, b in zip(self.model.buffers(), model.buffers()):
            eb.copy_(b)


def soft_ce(logits, t):
    return -(t * F.log_softmax(logits.float(), dim=1)).sum(1)


class Learner:
    """One network + optimiser + AMP scaler + EMA + cosine schedule (+ optional SAM)."""

    def __init__(self, args, device, seed, total_steps, warmup_steps):
        torch.manual_seed(seed)
        self.device = device
        self.model = PreActResNet18(NUM_CLASSES).to(device).to(memory_format=torch.channels_last)
        self.opt = build_optimizer(self.model, args)
        self.amp = device.type == 'cuda' and not args.no_amp
        self.scaler = torch.amp.GradScaler('cuda', enabled=self.amp)
        self.ema = EMA(self.model, args.ema) if args.ema > 0 else None
        self.rho = args.sam_rho
        self.total, self.warm, self.t = max(1, total_steps), max(1, warmup_steps), 0

    def lr_factor(self):
        if self.t < self.warm:
            return (self.t + 1) / self.warm
        p = min(1.0, (self.t - self.warm) / max(1, self.total - self.warm))
        return 0.5 * (1 + math.cos(math.pi * p))

    def current_lr(self):
        return self.opt.param_groups[-1]['lr']

    def _set_lr(self):
        f = self.lr_factor()
        for g in self.opt.param_groups:
            g['lr'] = g['base_lr'] * f

    def eval_model(self):
        return self.ema.model if self.ema is not None else self.model

    # --- standard step (soft targets, optional SAM) ---
    def _fwd_bwd(self, x, t):
        with autocast(self.device, self.amp):
            logits = self.model(x)
        loss = soft_ce(logits, t).mean()
        self.scaler.scale(loss).backward()
        return loss.detach()

    def train_step(self, x, t):
        self.model.train()
        self._set_lr()
        if self.rho > 0:
            loss = self._sam_step(x, t)
        else:
            loss = self._fwd_bwd(x, t)
            self.scaler.step(self.opt)
            self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        self._after_step()
        return loss

    def _sam_step(self, x, t):
        """SAM (Foret et al., 2021) compatible with the fp16 GradScaler: the perturbation
        rho * g / ||g|| is scale invariant, so the scaled gradients are used directly; on an
        overflow the perturbation is skipped and the scaler skips the step."""
        loss = self._fwd_bwd(x, t)
        params = [p for p in self.model.parameters() if p.grad is not None]
        with torch.no_grad():
            gnorm = torch.linalg.vector_norm(
                torch.stack([torch.linalg.vector_norm(p.grad.float()) for p in params]))
            coef = torch.where(torch.isfinite(gnorm), self.rho / (gnorm + 1e-12),
                               torch.zeros_like(gnorm))
            eps = []
            for p in params:
                e = torch.nan_to_num(p.grad * coef, nan=0.0, posinf=0.0, neginf=0.0)
                p.add_(e)
                eps.append(e)
        self.opt.zero_grad(set_to_none=True)
        with frozen_bn_stats(self.model):
            self._fwd_bwd(x, t)
        with torch.no_grad():
            for p, e in zip(params, eps):
                p.sub_(e)
        self.scaler.step(self.opt)
        self.scaler.update()
        return loss

    # --- split step, used by co-teaching (lets two GPUs overlap) ---
    def forward(self, x):
        self.model.train()
        self._set_lr()
        with autocast(self.device, self.amp):
            return self.model(x)

    def backward(self, loss):
        self.scaler.scale(loss).backward()

    def finish(self):
        self.scaler.step(self.opt)
        self.scaler.update()
        self.opt.zero_grad(set_to_none=True)
        self._after_step()

    def _after_step(self):
        if self.ema is not None:
            self.ema.update(self.model)
        self.t += 1


@torch.no_grad()
def predict(model, D, idx=None, tta=True, feats=False, bs=1024):
    """Softmax probabilities (and L2-normalised penultimate features) with flip-TTA."""
    model.eval()
    n = len(D) if idx is None else len(idx)
    amp = D.device.type == 'cuda'
    P, Fs = [], []
    for s in range(0, n, bs):
        ii = slice(s, min(n, s + bs)) if idx is None else idx[s:s + bs]
        x = D.batch(ii, augment=False)
        with autocast(D.device, amp):
            logit, f = model(x, feats=True)
            if tta:
                logit2, f2 = model(torch.flip(x, dims=[3]), feats=True)
        p = logit.float().softmax(1)
        if tta:
            p = 0.5 * (p + logit2.float().softmax(1))
            f = 0.5 * (f.float() + f2.float())
        P.append(p)
        if feats:
            Fs.append(F.normalize(f.float(), dim=1))
    P = torch.cat(P)
    return (P, torch.cat(Fs)) if feats else P


@torch.no_grad()
def bn_recalibrate(model, D, pool, steps=100, bs=256):
    """Recompute BatchNorm statistics for the (EMA) weights on augmented training data."""
    bns = [m for m in model.modules() if isinstance(m, nn.modules.batchnorm._BatchNorm)]
    saved = [m.momentum for m in bns]
    for m in bns:
        m.reset_running_stats()
        m.momentum = None
    model.train()
    amp = D.device.type == 'cuda'
    steps = min(steps, max(5, 2 * math.ceil(len(pool) / bs)))  # at most ~2 passes over the pool
    for _ in range(steps):
        idx = pool[torch.randint(len(pool), (bs,), device=pool.device)]
        with autocast(D.device, amp):
            model(D.batch(idx, augment=True))
    for m, mo in zip(bns, saved):
        m.momentum = mo
    model.eval()


def noisy_holdout_metrics(prob_fn, y_hold):
    if y_hold is None:
        return {}
    p = prob_fn()
    return dict(hold_acc=(p.argmax(1) == y_hold).float().mean().item(),
                hold_soft=p.gather(1, y_hold[:, None]).mean().item())


def _sync(devices):
    for d in devices:
        if d.type == 'cuda':
            torch.cuda.synchronize(d)


def time_it(fn, devices, warm=6, n=15):
    for _ in range(warm):
        fn()
    _sync(devices)
    t0 = time.perf_counter()
    for _ in range(n):
        fn()
    _sync(devices)
    return (time.perf_counter() - t0) / n


def eval_item_time(model, D, n=2048):
    idx = torch.arange(min(n, len(D)), device=D.device)
    return time_it(lambda: predict(model, D, idx, tta=False), [D.device], warm=1, n=2) / len(idx)


def plan_epochs(args, tag, step_sec, steps_per_epoch, eval_sec_per_epoch=0.0):
    epoch_sec = steps_per_epoch * step_sec + eval_sec_per_epoch
    if args.epochs > 0:
        E = args.epochs
    else:
        avail = args.budget_min * 60 - args.reserve_sec
        E = int(np.clip(int(avail / (1.04 * epoch_sec)), args.min_epochs, args.max_epochs))
        log(f'[{tag}] {step_sec * 1e3:.1f} ms/step, ~{epoch_sec:.1f} s/epoch -> {E} epochs '
            f'(~{E * epoch_sec / 60:.0f} min of the {args.budget_min:g} min budget)')
    return E


def free_cuda():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def bench_learner(args, dev, seed, batch_fn, eval_D=None):
    if args.epochs > 0:
        return 0.0, 0.0
    tmp = Learner(args, dev, seed, 10 ** 6, 1)
    step = time_it(lambda: tmp.train_step(*batch_fn()), [dev])
    item = eval_item_time(tmp.model, eval_D) if eval_D is not None else 0.0
    del tmp
    free_cuda()
    return step, item


class EpochLogger:
    def __init__(self, tag, total):
        self.tag, self.total, self.t0, self.rows = tag, total, time.time(), []

    def start(self):
        self.te = time.time()

    def end(self, rec):
        now = time.time()
        rec['sec'] = now - self.te
        self.rows.append(rec)
        e = rec['epoch']
        eta = (now - self.t0) / e * (self.total - e) / 60
        body = ' '.join(f'{k} {v:.4g}' if isinstance(v, float) else f'{k} {v}'
                        for k, v in rec.items() if k not in ('epoch',))
        log(f'[{self.tag}] ep {e}/{self.total} {body} | eta {eta:.1f} min')


def train_static(args, D, y, train_idx, hold_idx, seed, tag):
    """Plain training on fixed labels (cross-entropy + mixup + EMA, optional SAM / Muon)."""
    dev = D.device
    pool = torch.as_tensor(train_idx, device=dev)
    hold = torch.as_tensor(hold_idx, device=dev) if len(hold_idx) else None
    spe = max(1, len(pool) // args.bs)

    def rand_batch():
        return make_batch(D, pool[torch.randint(len(pool), (args.bs,), device=dev)], y, args.mixup)

    step_sec, item_sec = bench_learner(args, dev, seed, rand_batch, D if hold is not None else None)
    E = plan_epochs(args, tag, step_sec, spe, (len(hold) if hold is not None else 0) * item_sec)
    L = Learner(args, dev, seed, E * spe, args.lr_warmup_epochs * spe)
    lg = EpochLogger(tag, E)
    for e in range(E):
        lg.start()
        ls = torch.zeros((), device=dev)
        for idx in epoch_batches(pool, args.bs, spe):
            ls += L.train_step(*make_batch(D, idx, y, args.mixup))
        rec = dict(epoch=e + 1, lr=L.current_lr(), loss=ls.item() / spe, n_used=len(pool))
        rec.update(noisy_holdout_metrics(lambda: predict(L.eval_model(), D, hold, tta=False),
                                         y[hold] if hold is not None else None))
        lg.end(rec)
    return L, lg.rows, dict(epochs_planned=E)


def _gmm_low_posterior(x, iters=40):
    """2-component 1-D GMM (EM); posterior of the low-mean (clean) component."""
    x = x.astype(np.float64)
    lo, hi = x.min(), x.max()
    if hi - lo < 1e-8:
        return np.ones_like(x)
    x = (x - lo) / (hi - lo)
    mu = np.quantile(x, [0.25, 0.75])
    var = np.full(2, max(x.var(), 1e-3))
    pi = np.full(2, 0.5)
    for _ in range(iters):
        lp = -0.5 * (x[:, None] - mu) ** 2 / var - 0.5 * np.log(2 * np.pi * var) + np.log(pi + 1e-12)
        lp -= lp.max(1, keepdims=True)
        r = np.exp(lp)
        r /= r.sum(1, keepdims=True)
        nk = r.sum(0) + 1e-9
        pi = nk / nk.sum()
        mu = (r * x[:, None]).sum(0) / nk
        var = (r * (x[:, None] - mu) ** 2).sum(0) / nk + 5e-4
    return r[:, int(np.argmin(mu))]


def gmm_clean_prob(loss, labels, per_class=True, min_count=30):
    
    w = _gmm_low_posterior(loss)
    if per_class:
        for c in np.unique(labels):
            m = labels == c
            if m.sum() >= min_count:
                w[m] = _gmm_low_posterior(loss[m])
    return w


def train_gmm(args, D, y, train_idx, hold_idx, seed, tag, out_dir):

    dev = D.device
    pool = torch.as_tensor(train_idx, device=dev)
    yp = y[pool]
    yp_np = yp.cpu().numpy()
    hold = torch.as_tensor(hold_idx, device=dev) if len(hold_idx) else None
    spe = max(1, len(pool) // args.bs)
    T = torch.zeros(len(D), NUM_CLASSES, device=dev)
    T[pool] = F.one_hot(yp, NUM_CLASSES).float()

    def rand_batch():
        return make_batch(D, pool[torch.randint(len(pool), (args.bs,), device=dev)], T, args.mixup)

    step_sec, item_sec = bench_learner(args, dev, seed, rand_batch, D)
    E = plan_epochs(args, tag, step_sec, spe,
                    (len(pool) + (len(hold) if hold is not None else 0)) * item_sec)
    W = args.warm_epochs if args.warm_epochs > 0 else max(3, round(0.1 * E))
    snap_epochs = sorted({W, (W + E) // 2, E - 1})
    L = Learner(args, dev, seed, E * spe, args.lr_warmup_epochs * spe)
    lg = EpochLogger(tag, E)
    active, temporal, prev_action, snaps = pool, None, None, {}
    for e in range(E):
        lg.start()
        st = dict(n_clean=len(pool), n_relabel=0, n_drop=0)
        if e >= W:
            probs = predict(L.eval_model(), D, pool, tta=False)
            temporal = probs if temporal is None else args.te_beta * temporal + (1 - args.te_beta) * probs
            loss = -torch.log(probs.gather(1, yp[:, None]).squeeze(1).clamp_min(1e-8))
            w = torch.from_numpy(gmm_clean_prob(loss.cpu().numpy(), yp_np)).to(dev).float()
            conf, pl = temporal.max(1)
            clean = (w > args.p_clean) | ((conf >= args.tau_pl) & (pl == yp))
            relabel = ~clean & (conf >= args.tau_pl) & (pl != yp)
            onehot = F.one_hot(yp, NUM_CLASSES).float()
            T[pool] = torch.where(clean[:, None], w[:, None] * onehot + (1 - w[:, None]) * temporal,
                                  F.one_hot(pl, NUM_CLASSES).float())
            keep = clean | relabel
            active = pool[keep]
            action = torch.full((len(pool),), 2, dtype=torch.int8, device=dev)
            action[clean] = 0
            action[relabel] = 1
            st = dict(n_clean=int(clean.sum()), n_relabel=int(relabel.sum()), n_drop=int((~keep).sum()),
                      noise_est=float(1 - (w > args.p_clean).float().mean()),
                      changed=float((action != prev_action).float().mean()) if prev_action is not None else 0.0)
            prev_action = action
            if e in snap_epochs:
                snaps[f'loss_e{e}'] = loss.cpu().numpy().astype(np.float16)
                snaps[f'w_e{e}'] = w.cpu().numpy().astype(np.float16)
        ls = torch.zeros((), device=dev)
        for idx in epoch_batches(active, args.bs, spe):
            ls += L.train_step(*make_batch(D, idx, T, args.mixup))
        rec = dict(epoch=e + 1, lr=L.current_lr(), loss=ls.item() / spe, n_used=len(active), **st)
        rec.update(noisy_holdout_metrics(lambda: predict(L.eval_model(), D, hold, tta=False),
                                         y[hold] if hold is not None else None))
        lg.end(rec)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
        np.savez(os.path.join(out_dir, 'gmm_snapshots.npz'), labels=yp_np, **snaps)
        if prev_action is not None:
            np.savez(os.path.join(out_dir, 'final_division.npz'), idx=train_idx,
                     action=prev_action.cpu().numpy(), pseudo=temporal.argmax(1).cpu().numpy())
    return L, lg.rows, dict(epochs_planned=E, warm_epochs=W)


def coteach_step(L1, L2, D1, D2, y1, y2, idx_cpu, forget):
    i1 = idx_cpu.to(D1.device, non_blocking=True)
    i2 = idx_cpu.to(D2.device, non_blocking=True)
    o1 = L1.forward(D1.batch(i1, True))
    o2 = L2.forward(D2.batch(i2, True))
    l1 = F.cross_entropy(o1.float(), y1[i1], reduction='none')
    l2 = F.cross_entropy(o2.float(), y2[i2], reduction='none')
    k = max(1, int(round((1 - forget) * len(idx_cpu))))
    s1 = torch.topk(l1.detach(), k, largest=False).indices  # small-loss picks of net 1 ...
    s2 = torch.topk(l2.detach(), k, largest=False).indices
    loss1 = l1[s2.to(D1.device)].mean()  # ... train net 2, and vice versa
    loss2 = l2[s1.to(D2.device)].mean()
    L1.backward(loss1)
    L2.backward(loss2)
    L1.finish()
    L2.finish()
    return loss1.detach(), loss2.detach(), s1, s2


def train_coteach(args, data, y_np, train_idx, hold_idx, seed, tag):
    """Co-teaching (Han et al., 2018): two networks, each selects its (1-forget) fraction of
    small-loss samples in every mini-batch and the PEER network is updated on them.
    forget rate: 0 during warm-up, then ramps linearly to the noise rate estimated by a GMM on
    the losses at the end of warm-up. One network per GPU when two GPUs are available."""
    n_gpu = torch.cuda.device_count()
    if n_gpu >= 2:
        d1, d2 = torch.device('cuda:0'), torch.device('cuda:1')
    else:
        d1 = d2 = torch.device('cuda:0' if n_gpu else 'cpu')
    D1 = DeviceData(data['x_train'], d1)
    D2 = D1 if d2 == d1 else DeviceData(data['x_train'], d2)
    y1 = torch.as_tensor(y_np, device=d1)
    y2 = torch.as_tensor(y_np, device=d2)
    pool = torch.as_tensor(train_idx)  # CPU, broadcast to both devices per batch
    hold1 = torch.as_tensor(hold_idx, device=d1) if len(hold_idx) else None
    hold2 = torch.as_tensor(hold_idx, device=d2) if len(hold_idx) else None
    spe = max(1, len(pool) // args.bs)

    if args.epochs > 0:
        step_sec = item_sec = 0.0
    else:
        A, B = Learner(args, d1, seed, 10 ** 6, 1), Learner(args, d2, seed + 1, 10 ** 6, 1)
        step_sec = time_it(lambda: coteach_step(A, B, D1, D2, y1, y2,
                                                pool[torch.randint(len(pool), (args.bs,))], 0.3),
                           [d1, d2])
        item_sec = eval_item_time(A.model, D1)
        del A, B
        free_cuda()
    E = plan_epochs(args, tag, step_sec, spe, 2 * len(hold_idx) * item_sec)
    W = args.warm_epochs if args.warm_epochs > 0 else max(3, round(0.05 * E))
    L1 = Learner(args, d1, seed, E * spe, args.lr_warmup_epochs * spe)
    L2 = Learner(args, d2, seed + 1, E * spe, args.lr_warmup_epochs * spe)

    def both_probs(D_a, D_b, idx_a, idx_b, tta=False):
        return 0.5 * (predict(L1.eval_model(), D_a, idx_a, tta=tta) +
                      predict(L2.eval_model(), D_b, idx_b, tta=tta).to(D_a.device))

    lg = EpochLogger(tag, E)
    tau_hat = None
    for e in range(E):
        lg.start()
        if e == W:
            pool1 = pool.to(d1)
            probs = predict(L1.eval_model(), D1, pool1, tta=False)
            loss = -torch.log(probs.gather(1, y1[pool1][:, None]).squeeze(1).clamp_min(1e-8))
            w = gmm_clean_prob(loss.cpu().numpy(), y_np[train_idx])
            tau_hat = float(np.clip(1 - (w > 0.5).mean(), 0.05, 0.6))
            log(f'[{tag}] estimated noise rate after warm-up: {tau_hat:.3f}')
        forget = 0.0 if e < W else tau_hat * min(1.0, (e - W + 1) / args.ramp_epochs)
        perm = pool[torch.randperm(len(pool))]
        s_l1 = torch.zeros((), device=d1)
        s_l2 = torch.zeros((), device=d2)
        ov, n_ov = torch.zeros((), device=d1), 0
        for s in range(spe):
            idx = perm[s * args.bs:(s + 1) * args.bs]
            l1, l2, s1, s2 = coteach_step(L1, L2, D1, D2, y1, y2, idx, forget)
            s_l1 += l1
            s_l2 += l2
            if s % 25 == 0:
                m = torch.zeros(len(idx), dtype=torch.bool, device=d1)
                m[s1] = True
                ov += m[s2.to(d1)].float().mean()
                n_ov += 1
        rec = dict(epoch=e + 1, lr=L1.current_lr(), loss=s_l1.item() / spe, loss_net2=s_l2.item() / spe,
                   forget=forget, overlap=ov.item() / max(1, n_ov), n_used=int(round((1 - forget) * len(pool))))
        if hold1 is not None:
            rec.update(noisy_holdout_metrics(lambda: both_probs(D1, D2, hold1, hold2), y1[hold1]))
        lg.end(rec)
    return (L1, L2, D1, D2), lg.rows, dict(epochs_planned=E, warm_epochs=W, tau_hat=tau_hat)


class DiscountedUCB:
    """Discounted UCB (Garivier & Moulines, 2011) for a non-stationary multi-armed bandit."""

    def __init__(self, n_arms, gamma, c):
        self.n = np.zeros(n_arms)
        self.s = np.zeros(n_arms)
        self.tried = np.zeros(n_arms, bool)
        self.gamma, self.c = gamma, c

    def select(self):
        if not self.tried.all():
            return int(np.flatnonzero(~self.tried)[0])
        ucb = self.s / self.n + self.c * np.sqrt(max(np.log(self.n.sum()), 0.0) / self.n)
        return int(np.argmax(ucb))

    def update(self, a, r):
        self.n *= self.gamma
        self.s *= self.gamma
        self.n[a] += 1
        self.s[a] += r
        self.tried[a] = True

    def means(self):
        return np.where(self.n > 0, self.s / np.maximum(self.n, 1e-12), 0.0)


def classwise_rank(loss, y):
    """Percentile (0,1] of every sample's loss inside its own (given) class."""
    r = torch.empty_like(loss)
    for c in torch.unique(y):
        m = torch.nonzero(y == c).squeeze(1)
        order = torch.argsort(loss[m])
        rr = torch.empty(len(m), device=loss.device)
        rr[order] = torch.arange(1, len(m) + 1, device=loss.device, dtype=torch.float32) / len(m)
        r[m] = rr
    return r


def train_bandit(args, D, y, train_idx, hold_idx, seed, tag):
    if not len(hold_idx):
        raise SystemExit('--mode bandit needs --holdout > 0 (the reward is measured on it)')
    dev = D.device
    pool = torch.as_tensor(train_idx, device=dev)
    yp = y[pool]
    hold = torch.as_tensor(hold_idx, device=dev)
    arms = [float(a) for a in args.arms.split(',')]
    R = args.rounds
    spe = max(1, len(pool) // args.bs)
    spr = max(1, spe // R)

    def rand_batch():
        return make_batch(D, pool[torch.randint(len(pool), (args.bs,), device=dev)], y, args.mixup)

    step_sec, item_sec = bench_learner(args, dev, seed, rand_batch, D)
    E = plan_epochs(args, tag, step_sec, R * spr, (len(pool) + (R + 1) * len(hold)) * item_sec)
    W = args.warm_epochs if args.warm_epochs > 0 else max(3, round(0.1 * E))
    L = Learner(args, dev, seed, E * R * spr, args.lr_warmup_epochs * R * spr)
    bandit = DiscountedUCB(len(arms), args.ucb_gamma, args.ucb_c)
    yh = y[hold]

    def score():
        return predict(L.model, D, hold, tta=False).gather(1, yh[:, None]).mean().item()

    lg = EpochLogger(tag, E)
    history, prev, scale = [], None, None
    for e in range(E):
        lg.start()
        rank = None
        if e >= W:
            probs = predict(L.eval_model(), D, pool, tta=False)
            loss = -torch.log(probs.gather(1, yp[:, None]).squeeze(1).clamp_min(1e-8))
            rank = classwise_rank(loss, yp)
        ls = torch.zeros((), device=dev)
        ratios = []
        for rd in range(R):
            if rank is None:
                a, ratio, subset = None, 1.0, pool
            else:
                a = bandit.select()
                ratio = arms[a]
                subset = pool[rank <= ratio]
            for _ in range(spr):
                idx = subset[torch.randint(len(subset), (args.bs,), device=dev)]
                ls += L.train_step(*make_batch(D, idx, y, args.mixup))
            ratios.append(ratio)
            if e >= W - 1:
                cur = score()
                if a is not None and prev is not None:
                    gain = cur - prev
                    scale = abs(gain) + 1e-8 if scale is None else 0.9 * scale + 0.1 * abs(gain)
                    reward = gain / (scale + 1e-8)
                    bandit.update(a, reward)
                    history.append(dict(epoch=e + 1, round=rd, arm=a, ratio=ratio, gain=gain,
                                        reward=reward, score=cur))
                prev = cur
        rec = dict(epoch=e + 1, lr=L.current_lr(), loss=ls.item() / (R * spr), ratio=float(np.mean(ratios)),
                   n_used=int(np.mean(ratios) * len(pool)))
        rec.update(noisy_holdout_metrics(lambda: predict(L.eval_model(), D, hold, tta=False), yh))
        if e >= W:
            rec['arm_means'] = [round(float(v), 3) for v in bandit.means()]
        lg.end(rec)
    return L, lg.rows, dict(epochs_planned=E, warm_epochs=W, arms=arms, bandit_history=history)


def write_submission(path, pred):
    pred = np.asarray(pred).astype(int)
    with open(path, 'w') as f:
        f.write('ID,target\n')
        f.writelines(f'{i},{c}\n' for i, c in enumerate(pred))


def save_run(out_dir, args, rows, test_probs, extra):
    os.makedirs(out_dir, exist_ok=True)
    np.save(os.path.join(out_dir, 'test_probs.npy'), test_probs.astype(np.float16))
    write_submission(os.path.join(out_dir, 'submission.csv'), test_probs.argmax(1))
    with open(os.path.join(out_dir, 'metrics.json'), 'w') as f:
        json.dump(dict(args={k: v for k, v in vars(args).items() if k != 'fn'}, epochs=rows, **extra),
                  f, indent=1, default=_jsonable)
    log(f'saved {out_dir}/submission.csv  (prediction histogram: min {np.bincount(test_probs.argmax(1), minlength=NUM_CLASSES).min()}'
        f' / max {np.bincount(test_probs.argmax(1), minlength=NUM_CLASSES).max()} per class)')


def cmd_probe(args):
    data = load_data(args.root, args.cache)
    y_np = data['y_noisy']
    folds = stratified_folds(y_np, args.folds, args.fold_seed)
    ids = [int(s) for s in args.fold_ids.split(',')] if args.fold_ids else list(range(args.folds))
    dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    D, Dt = DeviceData(data['x_train'], dev), DeviceData(data['x_test'], dev)
    y = torch.as_tensor(y_np, device=dev)
    os.makedirs(args.out, exist_ok=True)
    for k in ids:
        t0 = time.time()
        seed = args.seed + 1000 * k
        seed_all(seed)
        train_idx, held = np.flatnonzero(folds != k), np.flatnonzero(folds == k)
        L, rows, extra = train_static(args, D, y, train_idx, [], seed, f'probe fold {k}')
        model = L.eval_model()
        bn_recalibrate(model, D, torch.as_tensor(train_idx, device=dev))
        p_held = predict(model, D, torch.as_tensor(held, device=dev), tta=True).cpu().numpy()
        _, feats = predict(model, D, None, tta=True, feats=True)
        test_p = predict(model, Dt, tta=True).cpu().numpy()
        agree = float((p_held.argmax(1) == y_np[held]).mean())
        np.savez(os.path.join(args.out, f'fold{k}.npz'), held_idx=held, probs=p_held.astype(np.float32),
                 feats=feats.cpu().numpy().astype(np.float16), test_probs=test_p.astype(np.float16),
                 log=json.dumps(dict(epochs=rows, oof_agreement_with_noisy=agree, minutes=(time.time() - t0) / 60,
                                     **extra), default=_jsonable))
        log(f'[probe fold {k}] out-of-fold agreement with the noisy labels: {agree:.4f} '
            f'({(time.time() - t0) / 60:.1f} min)')
        del L, model, feats
        free_cuda()
    with open(os.path.join(args.out, 'probe_meta.json'), 'w') as f:
        json.dump(dict(folds=args.folds, fold_seed=args.fold_seed), f)


def confident_learning(P, y, relabel_min):
    N, C = P.shape
    ar = np.arange(N)
    counts = np.bincount(y, minlength=C)
    thr = np.array([P[y == j, j].mean() if counts[j] else 1.0 for j in range(C)])
    above = P >= thr[None, :]
    has = above.any(1)
    yhat = np.where(above, P, -1.0).argmax(1)
    cj = np.zeros((C, C))
    np.add.at(cj, (y[has], yhat[has]), 1)
    cj = cj / np.maximum(cj.sum(1, keepdims=True), 1) * counts[:, None]
    cj *= N / cj.sum()
    n_issue = np.round(cj.sum(1) - np.diag(cj)).astype(int)
    Po = P.copy()
    Po[ar, y] = -1
    alt = Po.argmax(1)
    margin = P[ar, alt] - P[ar, y]
    flagged = np.zeros(N, bool)
    for j in range(C):
        idx = np.flatnonzero(y == j)
        k = int(np.clip(n_issue[j], 0, len(idx)))
        if k:
            flagged[idx[np.argsort(-margin[idx])[:k]]] = True
    new = np.where(has & (yhat != y), yhat, alt)
    relabel = flagged & (new != y) & (P[ar, new] >= relabel_min)
    drop = flagged & ~relabel
    labels = np.where(relabel, new, y)
    labels[drop] = -1
    action = np.zeros(N, np.int8)
    action[relabel] = 1
    action[drop] = 2
    return dict(labels=labels, action=action, cj=cj, thr=thr, noise_est=float(1 - np.trace(cj) / N))


def knn_votes(feats, helds, labels_cur, k, T, device):
    """Weighted kNN votes. Queries = held-out images of a fold, references = the images that
    fold's model was trained on (same feature space, query never seen by the model)."""
    N = len(labels_cur)
    V = np.zeros((N, NUM_CLASSES), np.float32)
    lab = torch.as_tensor(labels_cur, device=device)
    for Fk, held in zip(feats, helds):
        Ft = torch.as_tensor(Fk, device=device).float()
        is_ref = lab >= 0
        is_ref[torch.as_tensor(held, device=device)] = False
        ref = torch.nonzero(is_ref).squeeze(1)
        Fr, yr = Ft[ref], lab[ref]
        held_t = torch.as_tensor(held, device=device)
        for s in range(0, len(held_t), 1024):
            q = held_t[s:s + 1024]
            top, ti = (Ft[q] @ Fr.T).topk(min(k, len(ref)), dim=1)
            w = torch.exp((top - 1) / T)
            v = torch.zeros(len(q), NUM_CLASSES, device=device).scatter_add_(1, yr[ti], w)
            V[q.cpu().numpy()] = (v / v.sum(1, keepdim=True)).cpu().numpy()
        del Ft, Fr
    return V


def vote_decision(S, y, keep_ratio, relabel_min, agree=None):
    ar = np.arange(len(y))
    top = S.argmax(1)
    smax, sy = S[ar, top], S[ar, y]
    keep = sy >= keep_ratio * smax
    relabel = ~keep & (smax >= relabel_min) & (top != y)
    if agree is not None:
        relabel &= agree
    labels = np.where(keep, y, np.where(relabel, top, -1))
    action = np.where(keep, 0, np.where(relabel, 1, 2)).astype(np.int8)
    return labels, action


def _counts(action):
    return {a: int((action == i).sum()) for i, a in enumerate(ACTIONS)}


def cmd_clean(args):
    data = load_data(args.root, args.cache)
    y, classes = data['y_noisy'], data['classes']
    N = len(y)
    files = sorted(glob.glob(os.path.join(args.probe, 'fold*.npz')))
    if not files:
        raise SystemExit(f'no probe outputs in {args.probe}')
    P = np.zeros((N, NUM_CLASSES), np.float32)
    covered = np.zeros(N, bool)
    helds, feats = [], []
    for fp in files:
        z = np.load(fp)
        P[z['held_idx']] = z['probs']
        covered[z['held_idx']] = True
        helds.append(z['held_idx'])
        if args.method in ('knn', 'combo'):
            feats.append(z['feats'])
    if not covered.all():
        raise SystemExit(f'probe outputs cover {covered.sum()}/{N} samples: some folds are missing')
    dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

    cl = confident_learning(P, y, args.cl_relabel_min)
    stats = dict(method=args.method, n=N, oof_agreement_with_noisy=float((P.argmax(1) == y).mean()),
                 cl_noise_est=cl['noise_est'], cl=_counts(cl['action']))
    log(f'[clean] out-of-fold agreement with noisy labels {stats["oof_agreement_with_noisy"]:.4f}; '
        f'CL estimated noise rate {cl["noise_est"]:.4f}; CL actions {stats["cl"]}')
    out = dict(y_noisy=y, oof_probs_given=P[np.arange(N), y], oof_pred=P.argmax(1),
               oof_conf=P.max(1), cl_action=cl['action'], cl_cj=cl['cj'].astype(np.float32))
    cj = cl['cj'].copy()
    np.fill_diagonal(cj, 0)
    order = np.argsort(-cj, axis=None)[:25]
    stats['top_confusions'] = [dict(given=classes[i // NUM_CLASSES], likely=classes[i % NUM_CLASSES],
                                    count=float(cj.flat[i])) for i in order]
    diag = np.diag(cl['cj']) / np.maximum(cl['cj'].sum(1), 1e-9)
    stats['class_noise_est'] = {classes[j]: float(1 - diag[j]) for j in range(NUM_CLASSES)}

    if args.method == 'cl':
        labels, action = cl['labels'], cl['action']
    else:
        cur = y.copy()
        stats['knn_rounds'] = []
        for it in range(args.knn_iters):
            V = knn_votes(feats, helds, cur, args.knn_k, args.knn_t, dev)
            lab_it, act_it = vote_decision(V, y, args.keep_ratio, args.relabel_min)
            stats['knn_rounds'].append(dict(round=it + 1, **_counts(act_it),
                                            changed_vs_prev=float((lab_it != cur).mean())))
            log(f'[clean] kNN round {it + 1}: {_counts(act_it)}')
            cur = lab_it
        out.update(knn_action=act_it, knn_vote_given=V[np.arange(N), y], knn_top=V.argmax(1))
        stats['knn'] = _counts(act_it)
        if args.method == 'knn':
            labels, action = lab_it, act_it
        else:
            S = 0.5 * (P + V)
            labels, action = vote_decision(S, y, args.keep_ratio, args.relabel_min,
                                           agree=P.argmax(1) == V.argmax(1))
    stats['final'] = _counts(action)
    stats['relabel_fraction'] = float((action == 1).mean())
    stats['drop_fraction'] = float((action == 2).mean())
    log(f'[clean] {args.method}: {stats["final"]} -> {int((labels >= 0).sum())} training samples')
    if stats['drop_fraction'] > 0.6:
        log('[clean] WARNING: more than 60% of the data would be dropped - the probe is probably '
            'under-trained (raise PROBE_FOLD_MIN); CIFAR-100N fine has ~40% noise.')
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    np.savez(args.out, labels=labels.astype(np.int64), action=action, classes=np.array(classes),
             stats=json.dumps(stats, default=_jsonable), **out)


def cmd_train(args):
    seed_all(args.seed)
    t0 = time.time()
    data = load_data(args.root, args.cache)
    y_np = data['y_noisy'].copy() if args.labels == 'noisy' else np.load(args.labels)['labels']
    if len(y_np) != len(data['y_noisy']):
        raise SystemExit('label file does not match the training set')
    avail = np.flatnonzero(y_np >= 0)
    hold_idx = stratified_subset(avail, y_np[avail], args.holdout, args.seed + 77)
    train_idx = np.setdiff1d(avail, hold_idx)
    tag = args.tag or os.path.basename(os.path.normpath(args.out))
    log(f'[{tag}] mode {args.mode}, opt {args.opt}, sam_rho {args.sam_rho}, mixup {args.mixup}, '
        f'{len(train_idx)} training samples, {len(hold_idx)} noisy-holdout samples')
    extra = dict(mode=args.mode, n_train=int(len(train_idx)), n_holdout=int(len(hold_idx)))

    if args.mode == 'coteach':
        (L1, L2, D1, D2), rows, ex = train_coteach(args, data, y_np, train_idx, hold_idx, args.seed, tag)
        probs = []
        for L, D in ((L1, D1), (L2, D2)):
            m = L.eval_model()
            bn_recalibrate(m, D, torch.as_tensor(train_idx, device=D.device))
            probs.append(predict(m, DeviceData(data['x_test'], D.device), tta=True).cpu().numpy())
        test_probs = 0.5 * (probs[0] + probs[1])
        os.makedirs(args.out, exist_ok=True)
        for i, p in enumerate(probs, 1):
            np.save(os.path.join(args.out, f'test_probs_net{i}.npy'), p.astype(np.float16))
        ex['net_agreement_on_argmax'] = float((probs[0].argmax(1) == probs[1].argmax(1)).mean())
    else:
        dev = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
        D = DeviceData(data['x_train'], dev)
        y = torch.as_tensor(y_np, device=dev)
        if args.mode == 'static':
            L, rows, ex = train_static(args, D, y, train_idx, hold_idx, args.seed, tag)
        elif args.mode == 'gmm':
            L, rows, ex = train_gmm(args, D, y, train_idx, hold_idx, args.seed, tag, args.out)
        elif args.mode == 'bandit':
            L, rows, ex = train_bandit(args, D, y, train_idx, hold_idx, args.seed, tag)
        else:
            raise SystemExit(f'unknown mode {args.mode}')
        m = L.eval_model()
        pool = torch.as_tensor(train_idx, device=dev)
        bn_recalibrate(m, D, pool)
        if len(hold_idx):
            h = torch.as_tensor(hold_idx, device=dev)
            ex['final_holdout'] = noisy_holdout_metrics(lambda: predict(m, D, h, tta=True), y[h])
        test_probs = predict(m, DeviceData(data['x_test'], dev), tta=True).cpu().numpy()
    extra.update(ex)
    extra['minutes'] = (time.time() - t0) / 60
    save_run(args.out, args, rows, test_probs, extra)
    log(f'[{tag}] done in {extra["minutes"]:.1f} min')


def cmd_ensemble(args):
    ps = []
    for path in args.inputs:
        p = np.load(path)['test_probs'] if path.endswith('.npz') else np.load(path)
        ps.append(p.astype(np.float32))
    w = [float(v) for v in args.weights.split(',')] if args.weights else [1.0] * len(ps)
    P = sum(wi * p for wi, p in zip(w, ps)) / sum(w)
    os.makedirs(os.path.dirname(os.path.abspath(args.out)), exist_ok=True)
    np.save(os.path.splitext(args.out)[0] + '_probs.npy', P.astype(np.float16))
    write_submission(args.out, P.argmax(1))
    agree = [float((p.argmax(1) == P.argmax(1)).mean()) for p in ps]
    log(f'ensemble of {len(ps)} -> {args.out}; agreement of each member with the ensemble: {agree}')


def _plt():
    import matplotlib.pyplot as plt
    plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.3})
    return plt


def load_run(run_dir):
    with open(os.path.join(run_dir, 'metrics.json')) as f:
        return json.load(f)


def _save(fig, save):
    if save:
        os.makedirs(os.path.dirname(os.path.abspath(save)), exist_ok=True)
        fig.savefig(save, bbox_inches='tight')


def plot_curves(runs, keys=('loss', 'hold_acc', 'n_used', 'lr'), save=None):
    plt = _plt()
    loaded = {k: load_run(v) for k, v in runs.items()}
    keys = [k for k in keys if any(k in r for m in loaded.values() for r in m['epochs'])]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.4 * len(keys), 3.3), squeeze=False)
    for label, m in loaded.items():
        ep = [r['epoch'] for r in m['epochs']]
        for ax, k in zip(axes[0], keys):
            ax.plot(ep, [r.get(k, np.nan) for r in m['epochs']], label=label)
    names = dict(loss='training loss', hold_acc='noisy-holdout accuracy', n_used='samples used / epoch',
                 lr='learning rate', forget='forget rate', ratio='keep ratio')
    for ax, k in zip(axes[0], keys):
        ax.set_title(names.get(k, k))
        ax.set_xlabel('epoch')
    axes[0][0].legend(fontsize=8)
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def run_table(runs):
    import pandas as pd
    rows = []
    for label, d in runs.items():
        m = load_run(d)
        a = m['args']
        last = m['epochs'][-1] if m['epochs'] else {}
        rows.append(dict(run=label, mode=m.get('mode'), optimizer=a.get('opt'), sam_rho=a.get('sam_rho'),
                         mixup=a.get('mixup'), labels=os.path.basename(str(a.get('labels'))),
                         train_samples=m.get('n_train'), epochs=len(m['epochs']),
                         minutes=round(m.get('minutes', float('nan')), 1),
                         final_train_loss=round(last.get('loss', float('nan')), 3),
                         noisy_holdout_acc=(m.get('final_holdout') or {}).get('hold_acc')))
    return pd.DataFrame(rows)


def clean_summary(npz_path):
    import pandas as pd
    z = np.load(npz_path)
    s = json.loads(str(z['stats']))
    print(f"method={s['method']}  out-of-fold agreement with noisy labels={s['oof_agreement_with_noisy']:.4f}  "
          f"CL estimated noise rate={s['cl_noise_est']:.4f}")
    print('final actions:', s['final'], f"(relabel {s['relabel_fraction']:.1%}, drop {s['drop_fraction']:.1%})")
    if 'knn_rounds' in s:
        print(pd.DataFrame(s['knn_rounds']).to_string(index=False))
    return pd.DataFrame(s['top_confusions']).head(15)


def plot_clean(npz_path, save=None):
    plt = _plt()
    z = np.load(npz_path)
    s = json.loads(str(z['stats']))
    action, y = z['action'], z['y_noisy']
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    cnt = [int((action == i).sum()) for i in range(3)]
    bars = ax[0].bar(ACTIONS, cnt, color=['#4C78A8', '#F58518', '#E45756'])
    for b, c in zip(bars, cnt):
        ax[0].text(b.get_x() + b.get_width() / 2, c, f'{c}\n{c / len(action):.1%}', ha='center', va='bottom', fontsize=8)
    ax[0].set_title(f"{s['method']}: decision per training image")
    ax[0].set_ylim(0, max(cnt) * 1.25)
    noise = np.array(list(s['class_noise_est'].values()))
    names = list(s['class_noise_est'].keys())
    o = np.argsort(-noise)
    ax[1].bar(range(NUM_CLASSES), noise[o], color='#72B7B2')
    ax[1].axhline(s['cl_noise_est'], color='k', ls='--', lw=1, label=f"overall {s['cl_noise_est']:.1%}")
    for r in range(5):
        ax[1].text(r, noise[o[r]], names[o[r]], rotation=90, fontsize=7, va='bottom')
    ax[1].set_title('estimated noise rate per given class (confident joint)')
    ax[1].set_xlabel('class (sorted)')
    ax[1].legend(fontsize=8)
    pg = z['oof_probs_given']
    for i, a in enumerate(ACTIONS):
        m = action == i
        if m.any():
            ax[2].hist(pg[m], bins=50, range=(0, 1), alpha=0.6, label=f'{a} ({m.sum()})')
    ax[2].set_title('out-of-fold p(given label) by decision')
    ax[2].set_yscale('log')
    ax[2].legend(fontsize=8)
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def plot_agreement(npz_path, a='cl_action', b='knn_action', save=None):
    plt = _plt()
    z = np.load(npz_path)
    A, B = z[a], z[b]
    M = np.array([[int(((A == i) & (B == j)).sum()) for j in range(3)] for i in range(3)])
    fig, ax = plt.subplots(figsize=(4.4, 3.8))
    ax.imshow(M, cmap='Blues')
    ax.grid(False)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{M[i, j]}\n{M[i, j] / M.sum():.1%}', ha='center', va='center', fontsize=8,
                    color='white' if M[i, j] > M.max() / 2 else 'black')
    ax.set_xticks(range(3), ACTIONS)
    ax.set_yticks(range(3), ACTIONS)
    ax.set_xlabel(b.replace('_action', ''))
    ax.set_ylabel(a.replace('_action', ''))
    same_label = float(((A == B)).mean())
    ax.set_title(f'decision agreement ({same_label:.1%} identical)')
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def plot_gmm(run_dir, save=None):
    plt = _plt()
    m = load_run(run_dir)
    rows = [r for r in m['epochs'] if 'noise_est' in r]
    snaps = np.load(os.path.join(run_dir, 'gmm_snapshots.npz'))
    keys = sorted([k for k in snaps.files if k.startswith('loss_e')], key=lambda k: int(k[6:]))
    fig, ax = plt.subplots(1, 1 + len(keys), figsize=(4.4 * (1 + len(keys)), 3.4), squeeze=False)
    ax = ax[0]
    if rows:
        ep = [r['epoch'] for r in rows]
        ax[0].stackplot(ep, [r['n_clean'] for r in rows], [r['n_relabel'] for r in rows],
                        [r['n_drop'] for r in rows], labels=['clean (given label)', 'relabeled', 'excluded'],
                        colors=['#4C78A8', '#F58518', '#E45756'], alpha=0.85)
        ax[0].legend(fontsize=7, loc='lower left')
        ax2 = ax[0].twinx()
        ax2.plot(ep, [r['changed'] for r in rows], 'k:', lw=1)
        ax2.set_ylabel('fraction of decisions changed', fontsize=8)
    ax[0].set_title('dynamic division per epoch')
    ax[0].set_xlabel('epoch')
    for a, k in zip(ax[1:], keys):
        e = k[6:]
        loss, w = snaps[k].astype(np.float32), snaps[f'w_e{e}'].astype(np.float32)
        a.hist(loss[w > 0.5], bins=60, alpha=0.6, label='GMM clean')
        a.hist(loss[w <= 0.5], bins=60, alpha=0.6, label='GMM noisy')
        a.set_title(f'per-sample loss, epoch {int(e) + 1}')
        a.set_yscale('log')
        a.legend(fontsize=7)
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def plot_bandit(run_dir, save=None):
    plt = _plt()
    m = load_run(run_dir)
    h = m.get('bandit_history', [])
    arms = m['arms']
    if not h:
        print('no bandit decisions recorded')
        return
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
    x = np.arange(len(h))
    ax[0].scatter(x, [r['ratio'] for r in h], c=[r['epoch'] for r in h], s=6, cmap='viridis')
    k = max(1, len(h) // 25)
    ax[0].plot(x, np.convolve([r['ratio'] for r in h], np.ones(k) / k, mode='same'), 'r-', lw=1.5, label='moving avg')
    ax[0].set_title('keep-ratio chosen by the bandit')
    ax[0].set_xlabel('decision (1/R epoch)')
    ax[0].legend(fontsize=8)
    cnt = [sum(r['arm'] == i for r in h) for i in range(len(arms))]
    ax[1].bar([str(a) for a in arms], cnt, color='#54A24B')
    ax[1].set_title('how often each keep-ratio was played')
    ep = [r['epoch'] for r in m['epochs']]
    ax[2].plot(ep, [r.get('hold_acc', np.nan) for r in m['epochs']], label='noisy-holdout acc (EMA)')
    ax[2].plot(ep, [r.get('ratio', np.nan) for r in m['epochs']], label='mean keep-ratio')
    ax[2].set_xlabel('epoch')
    ax[2].legend(fontsize=8)
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def plot_coteach(run_dir, save=None):
    plt = _plt()
    m = load_run(run_dir)
    ep = [r['epoch'] for r in m['epochs']]
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
    ax[0].plot(ep, [r['forget'] for r in m['epochs']])
    ax[0].set_title(f"forget rate (estimated noise {m.get('tau_hat') or float('nan'):.3f})")
    ax[1].plot(ep, [r['overlap'] for r in m['epochs']])
    ax[1].set_title("overlap of the two networks' small-loss selections")
    ax[2].plot(ep, [r['loss'] for r in m['epochs']], label='net 1')
    ax[2].plot(ep, [r['loss_net2'] for r in m['epochs']], label='net 2')
    ax[2].set_title('loss on the peer-selected samples')
    ax[2].legend(fontsize=8)
    for a in ax:
        a.set_xlabel('epoch')
    fig.tight_layout()
    _save(fig, save)
    plt.show()


def check_submission(path, n_test=10000):
    import pandas as pd
    df = pd.read_csv(path)
    ok = (list(df.columns) == ['ID', 'target'] and len(df) == n_test and
          (df['ID'].values == np.arange(n_test)).all() and df['target'].between(0, NUM_CLASSES - 1).all())
    print(f'{path}: {len(df)} rows, {df["target"].nunique()} distinct classes, format ok = {ok}')
    return ok


def build_parser():
    base = argparse.ArgumentParser(add_help=False)
    base.add_argument('--root', default=DEFAULT_ROOT)
    base.add_argument('--cache', default='/tmp/nl_cache')
    base.add_argument('--seed', type=int, default=0)

    opt = argparse.ArgumentParser(add_help=False)
    opt.add_argument('--bs', type=int, default=128)
    opt.add_argument('--lr', type=float, default=0.1)
    opt.add_argument('--wd', type=float, default=5e-4)
    opt.add_argument('--opt', choices=['sgd', 'muon'], default='sgd')
    opt.add_argument('--muon-lr', type=float, default=0.02)
    opt.add_argument('--muon-wd', type=float, default=1e-3, help='decoupled: w *= 1 - lr*wd')
    opt.add_argument('--sam-rho', type=float, default=0.0, help='>0 enables SAM')
    opt.add_argument('--mixup', type=float, default=1.0, help='mixup alpha, 0 disables')
    opt.add_argument('--ema', type=float, default=0.999, help='0 disables the weight EMA')
    opt.add_argument('--epochs', type=int, default=0, help='0 = derive from --budget-min')
    opt.add_argument('--budget-min', type=float, default=40)
    opt.add_argument('--min-epochs', type=int, default=15)
    opt.add_argument('--max-epochs', type=int, default=300)
    opt.add_argument('--reserve-sec', type=float, default=60)
    opt.add_argument('--lr-warmup-epochs', type=float, default=2)
    opt.add_argument('--no-amp', action='store_true')

    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = p.add_subparsers(dest='cmd', required=True)

    sp = sub.add_parser('probe', parents=[base, opt])
    sp.add_argument('--out', default='runs/probe')
    sp.add_argument('--folds', type=int, default=4)
    sp.add_argument('--fold-ids', default='', help='comma separated subset, e.g. 0,1')
    sp.add_argument('--fold-seed', type=int, default=2024)
    sp.set_defaults(fn=cmd_probe)

    sc = sub.add_parser('clean', parents=[base])
    sc.add_argument('--probe', default='runs/probe')
    sc.add_argument('--method', choices=['cl', 'knn', 'combo'], required=True)
    sc.add_argument('--out', required=True)
    sc.add_argument('--cl-relabel-min', type=float, default=0.4)
    sc.add_argument('--knn-k', type=int, default=20)
    sc.add_argument('--knn-t', type=float, default=0.1)
    sc.add_argument('--knn-iters', type=int, default=2)
    sc.add_argument('--keep-ratio', type=float, default=0.5)
    sc.add_argument('--relabel-min', type=float, default=0.4)
    sc.set_defaults(fn=cmd_clean)

    st = sub.add_parser('train', parents=[base, opt])
    st.add_argument('--out', required=True)
    st.add_argument('--tag', default='')
    st.add_argument('--labels', default='noisy', help="'noisy' or a .npz written by the clean stage")
    st.add_argument('--mode', choices=['static', 'gmm', 'coteach', 'bandit'], default='static')
    st.add_argument('--holdout', type=int, default=0, help='noisy-label holdout taken from the training set')
    st.add_argument('--warm-epochs', type=int, default=0, help='0 = automatic')
    st.add_argument('--p-clean', type=float, default=0.5)
    st.add_argument('--tau-pl', type=float, default=0.6)
    st.add_argument('--te-beta', type=float, default=0.7, help='temporal ensemble momentum')
    st.add_argument('--ramp-epochs', type=int, default=10)
    st.add_argument('--arms', default='0.5,0.6,0.7,0.8,0.9,1.0')
    st.add_argument('--rounds', type=int, default=4, help='bandit decisions per epoch')
    st.add_argument('--ucb-c', type=float, default=0.8)
    st.add_argument('--ucb-gamma', type=float, default=0.97)
    st.set_defaults(fn=cmd_train)

    se = sub.add_parser('ensemble')
    se.add_argument('--inputs', nargs='+', required=True)
    se.add_argument('--weights', default='')
    se.add_argument('--out', required=True)
    se.set_defaults(fn=cmd_ensemble)
    return p


def main(argv=None):
    torch.backends.cudnn.benchmark = True
    args = build_parser().parse_args(argv)
    args.fn(args)


if __name__ == '__main__':
    main()

In [ ]:
import os, sys, subprocess, shlex, time, shutil, glob
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
os.chdir(WORK)
ROOT = '/kaggle/input/competitions/dl-gen-ai-2026-fii-competition-3/fii-atnn-2024-project-noisy-cifar-100'
COMMON = f'--root {ROOT} --cache /tmp/nl_cache'   # nl.py falls back to searching /kaggle/input


def n_gpus():
    try:
        out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout
    except FileNotFoundError:
        return 0
    return sum(l.startswith('GPU') for l in out.splitlines())


NGPU = n_gpus()
print(f'{NGPU} GPU(s) visible')
if NGPU:
    print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                         capture_output=True, text=True).stdout)
if NGPU < 2:
    print('WARNING: select the "GPU T4 x2" accelerator. With one GPU the parallel jobs share it '
          '(the epoch planner then simply gives each job fewer epochs).')


def run_jobs(jobs, poll=90):
    """jobs = [(name, gpu_ids, 'nl.py arguments')]; runs them in parallel, one progress line per job."""
    os.makedirs('logs', exist_ok=True)
    procs, t0 = [], time.time()
    for name, gpus, args in jobs:
        env = dict(os.environ, PYTHONUNBUFFERED='1')
        if NGPU:
            env['CUDA_VISIBLE_DEVICES'] = ','.join(sorted({str(int(g) % NGPU) for g in str(gpus).split(',')}))
        f = open(f'logs/{name}.log', 'w')
        procs.append((name, subprocess.Popen([sys.executable, 'nl.py'] + shlex.split(args),
                                             stdout=f, stderr=subprocess.STDOUT, env=env), f))
        print(f'started {name} on GPU {env.get("CUDA_VISIBLE_DEVICES", "cpu")}: nl.py {args}')

    def last(name):
        lines = open(f'logs/{name}.log').read().strip().splitlines()
        return lines[-1] if lines else ''

    while any(p.poll() is None for _, p, _ in procs):
        time.sleep(poll)
        for name, p, _ in procs:
            print(f'[{(time.time() - t0) / 60:5.1f} min] {name}: {last(name)}', flush=True)
    failed = []
    for name, p, f in procs:
        f.close()
        if p.returncode:
            failed.append(name)
            print(f'----- {name} FAILED -----\n' + open(f'logs/{name}.log').read()[-4000:])
    if failed:
        raise RuntimeError(f'failed jobs: {failed}')
    print(f'all jobs finished in {(time.time() - t0) / 60:.1f} min')

## Time budget (minutes)
The epoch count of every job is computed from these budgets after a short speed benchmark.

In [ ]:
TRAIN_MIN = 60

In [ ]:
run_jobs([('coteach', '0,1', f'train {COMMON} --mode coteach --labels noisy --mixup 0 --budget-min {TRAIN_MIN} --out runs/coteach')])

In [ ]:
%matplotlib inline
import importlib, nl
importlib.reload(nl)
os.makedirs('figs', exist_ok=True)
nl.plot_coteach('runs/coteach', save='figs/p2a_coteach.png')
nl.plot_curves({'co-teaching': 'runs/coteach'}, keys=('loss', 'n_used', 'lr'), save='figs/p2a_curves.png')
print('estimated noise rate:', nl.load_run('runs/coteach')['tau_hat'],
      '| agreement of the two nets on test argmax:', nl.load_run('runs/coteach')['net_agreement_on_argmax'])
nl.run_table({'co-teaching': 'runs/coteach'})